## Prepping Data

In [1]:
import os

# Tell transformers not to use TensorFlow (we only need PyTorch here)
os.environ["TRANSFORMERS_NO_TF"] = "1"

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

import transformers, sys, importlib

print("Python executable:", sys.executable)
print("Transformers version:", transformers.__version__)
print("TensorFlow present?", importlib.util.find_spec("tensorflow") is not None)
print("Keras present?", importlib.util.find_spec("keras") is not None)
print("tf_keras present?", importlib.util.find_spec("tf_keras") is not None)

/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Python executable: /Users/chadadelman/anaconda3/envs/chad_env/bin/python
Transformers version: 4.57.3
TensorFlow present? True
Keras present? True
tf_keras present? True


In [2]:
#We will need the training data text summary pairs
%store -r train_paired_summaries


In [3]:
#create object to be passed in as batch
#Needs to be dictinionary of lists
#Will be only training data
text_list=[]
summary_list=[]

for v in train_paired_summaries.values():
    text_list.append(v[0])
    summary_list.append(v[1])

training_batch = dict()
training_batch["text"] = text_list
training_batch["summary"] = summary_list

training_batch_dataset = Dataset.from_dict(training_batch)

## Model Fine Tuning

In [4]:
import torch
from torch.utils.data import DataLoader

def run_model(
        test_text_list
        ,learning_rate = 5e-7
        ,batch_size = 2
        ,num_epochs = 2
        ,test_size = 0.4
        ,max_input_length = 64
        ,max_target_length = 32
        ,max_length = 32
        ,num_beams = 2
        ,display_progress = True
):
    # Simple train/validation split
    dataset = training_batch_dataset.train_test_split(test_size=test_size, seed=42)
    train_dataset = dataset["train"]
    eval_dataset = dataset["test"]

    model_name = "t5-small"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    print("Tokenizer type:", type(tokenizer))
    print("Model type:", type(model))

    def preprocess_function(batch):
        # T5 likes a task prefix, e.g. "summarize: "
        inputs = ["summarize: " + t for t in batch["text"]]
        model_inputs = tokenizer(
            inputs,
            max_length=max_input_length,
            truncation=True,
            padding="max_length",
        )

        labels = tokenizer(
            batch["summary"],
            max_length=max_target_length,
            truncation=True,
            padding="max_length",
        )

        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    tokenized_train = train_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["text", "summary"],
    )

    tokenized_eval = eval_dataset.map(
        preprocess_function,
        batched=True,
        remove_columns=["text", "summary"],
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # DataLoader for training
    train_dataloader = DataLoader(
        tokenized_train,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=data_collator,
    )

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

    model.train()
    for epoch in range(num_epochs):
        total_loss = 0.0
        for step, batch in enumerate(train_dataloader):
            # Move batch to device
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(**batch)
            loss = outputs.loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

            if display_progress and (step + 1) % 25 == 0:
                print(f"Epoch {epoch+1}, Step {step+1}, Loss: {loss.item():.4f}")

        avg_loss = total_loss / len(train_dataloader)
        
        print(f"Epoch {epoch+1} finished. Average loss: {avg_loss:.4f}")

    model.eval()

    generated_text_list=[]

    for test_text in test_text_list:
        #summarize each item in test_text_list
        inputs = tokenizer(
            "summarize: " + test_text,
            return_tensors="pt",
            truncation=True,
            padding=True,
        ).to(device)

        with torch.no_grad():
            generated_ids = model.generate(
                **inputs,
                max_length=max_length,
                num_beams=num_beams,
            )

        generated_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        generated_text_list.append(generated_text)
    
    return generated_text_list


In [5]:
%store -r val_paired_summaries
#Dictinory with the key being the text name and the value being a 3 element tuple of (text, summary, human-made flag) where human-made flag is true if the summary was made by a human


In [6]:
#Looking at data
i=0
for k, v in val_paired_summaries.items():
    print(k, end=" | ")

    if i>300:
        break
    i+=1

print("\n")
text = val_paired_summaries["Hamlet-Act I-Scene I"][0]
#print(text)

Hamlet-Act I-Scene I | Hamlet-Act I-Scene II | Hamlet-Act I-Scene III | Hamlet-Act I-Scene IV | Hamlet-Act I-Scene V | Hamlet-Act II-Scene I | Hamlet-Act II-Scene II | Hamlet-Act III-Scene I | Hamlet-Act III-Scene II | Hamlet-Act III-Scene III | Hamlet-Act III-Scene IV | Hamlet-Act IV-Scene I | Hamlet-Act IV-Scene II | Hamlet-Act IV-Scene III | Hamlet-Act IV-Scene IV | Hamlet-Act IV-Scene V | Hamlet-Act IV-Scene VI | Hamlet-Act IV-Scene VII | Hamlet-Act V-Scene I | Hamlet-Act V-Scene II | Othello-Act I-Scene I | Othello-Act I-Scene II | Othello-Act I-Scene III | Othello-Act II-Scene I | Othello-Act II-Scene II | Othello-Act II-Scene III | Othello-Act III-Scene I | Othello-Act III-Scene II | Othello-Act III-Scene III | Othello-Act III-Scene IV | Othello-Act IV-Scene I | Othello-Act IV-Scene II | Othello-Act IV-Scene III | Othello-Act V-Scene I | Othello-Act V-Scene II | Romeo_and_Juliet-Act I-Scene I | Romeo_and_Juliet-Act I-Scene II | Romeo_and_Juliet-Act I-Scene III | Romeo_and_Juliet

In [7]:
#Test with 8 valdiation text 
#do human check (judged by me) for decent summarization
test_text_list=list()

test_text_list.append(val_paired_summaries["Hamlet-Act I-Scene I"][0])
test_text_list.append(val_paired_summaries["Hamlet-Act III-Scene IV"][0])

test_text_list.append(val_paired_summaries["Romeo_and_Juliet-Act I-Scene V"][0])
test_text_list.append(val_paired_summaries["Romeo_and_Juliet-Act II-Scene VI"][0])

test_text_list.append(val_paired_summaries["Othello-Act I-Scene I"][0])
test_text_list.append(val_paired_summaries["Othello-Act IV-Scene II"][0])

test_text_list.append(val_paired_summaries["Sonnet 67"][0])
test_text_list.append(val_paired_summaries["Sonnet 101"][0])


In [8]:
#Setting hyperparameters and testing
generated_text_1 = run_model(
        test_text_list = test_text_list
        ,learning_rate = 5e-7
        ,batch_size = 2
        ,num_epochs = 2
        ,test_size = 0.4
        ,max_input_length = 64
        ,max_target_length = 32
        ,max_length = 32
        ,num_beams = 2
        ,display_progress = True
)

Tokenizer type: <class 'transformers.models.t5.tokenization_t5_fast.T5TokenizerFast'>
Model type: <class 'transformers.models.t5.modeling_t5.T5ForConditionalGeneration'>


Map:   0%|          | 0/454 [00:00<?, ? examples/s]

Map:   0%|          | 0/303 [00:00<?, ? examples/s]

Epoch 1, Step 25, Loss: 5.8873
Epoch 1, Step 50, Loss: 5.3348
Epoch 1, Step 75, Loss: 5.1035
Epoch 1, Step 100, Loss: 5.6723
Epoch 1, Step 125, Loss: 5.4242
Epoch 1, Step 150, Loss: 4.8194
Epoch 1, Step 175, Loss: 6.8712
Epoch 1, Step 200, Loss: 5.4348
Epoch 1, Step 225, Loss: 5.2616
Epoch 1 finished. Average loss: 5.5181
Epoch 2, Step 25, Loss: 5.6894
Epoch 2, Step 50, Loss: 6.2865
Epoch 2, Step 75, Loss: 5.4692
Epoch 2, Step 100, Loss: 6.0307
Epoch 2, Step 125, Loss: 6.1001
Epoch 2, Step 150, Loss: 5.3522
Epoch 2, Step 175, Loss: 7.1417
Epoch 2, Step 200, Loss: 4.9857
Epoch 2, Step 225, Loss: 4.8003
Epoch 2 finished. Average loss: 5.4825
SUMMARIES: ["'Tis now struck twelve. Get thee to bed, Francisco. FRANCISCO. For this relief much thanks. BARNARDO", 'QUEEN. hamlet, thou hast thy father much offended. hamlet. hamlet. ham', "'tis gone, ’tis gone, ’tis gone, ’tis gone, ’tis gone, ", 'ROMEO. if the measure of thy joy Be heap’d like mine, then sweeten with thy breath This neighbour air'

In [9]:
print("Summary: ",end="")
print(*generated_text_1,sep="\nSummary: ")

Summary: 'Tis now struck twelve. Get thee to bed, Francisco. FRANCISCO. For this relief much thanks. BARNARDO
Summary: QUEEN. hamlet, thou hast thy father much offended. hamlet. hamlet. ham
Summary: 'tis gone, ’tis gone, ’tis gone, ’tis gone, ’tis gone, 
Summary: ROMEO. if the measure of thy joy Be heap’d like mine, then sweeten with thy breath This neighbour air
Summary: arithmetic expert says he has chosen his lieutenant. he says he has chosen his lieutenant. 
Summary: EMILIA. Never, my lord. OTHELLO. To fetch her fan, her gloves, her mask, nor nothing?
Summary: false painting imitates his cheek, and steals dead seeming of his living hue? why should he live, now Nature bankrupt is,
Summary: truant Muse: 'Truth needs no colour, with his colour fixed; Beauty no pencil, beauty's truth to lay; but
